# wildfire training notebook

This notebook performs training only: **verify data → preprocess → engineer features → split chronologically → fit → calibrate → evaluate → explain → export weights**.

Choose one data2.0 four-file dataset in the setup cell (defaults to lakshay654 Kaggle Inputs):

- `stage_c_knn` (default) → `/kaggle/input/datasets/lakshay654/california-wildfire-knn`
- `stage_c` → `/kaggle/input/datasets/lakshay654/california-wildfire-median`

Optionally restrict cells with `CELL_SUBSET` using `firms_test` / `fire_analysis2.csv`:

- `all` (default): full Stage C grid;
- `high_fire`: `High Outlier` + `High` (~25% of categorized cells);
- `high_medium_fire`: those plus `Medium` (~75% of categorized cells).

CSV path: `/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv` (needed only when subset ≠ `all`).

The output `models/champion_model.joblib` is the complete handoff to the separate inference notebook. It contains fitted preprocessing, classifier, ranker, probability calibrator, feature order, source-stage contract, cell subset, and blend weights. Plain-text LightGBM weight files are also exported for inspection.

```text
<four-file data2.0 dataset>/
├── all.parquet
├── meta.json
├── dataset_metadata.json
└── feature_columns.json
```

In Kaggle, attach the matching Input dataset(s), enable a GPU, choose **Run all**, then save the notebook output as a dataset so the inference notebook can attach it.


## 1. Training configuration, data discovery, and GPU

Defaults (edit only the stage / subset switches unless paths differ):

| Switch | Kaggle Input |
|---|---|
| `TRAINING_DATA_STAGE = "stage_c_knn"` | `/kaggle/input/datasets/lakshay654/california-wildfire-knn` |
| `TRAINING_DATA_STAGE = "stage_c"` | `/kaggle/input/datasets/lakshay654/california-wildfire-median` |
| `CELL_SUBSET` ≠ `all` | also attach `/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv` |

Enable a GPU accelerator for full training runs.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "champion-wildfire-mpl"))
os.environ.setdefault("MPLBACKEND", "Agg")

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import sklearn
from sklearn import set_config
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

SEED = 42
# Kaggle Input datasets (lakshay654):
#   california-wildfire-knn    → stage_c_knn
#   california-wildfire-median → stage_c
#   firms-test/fire_analysis2.csv → CELL_SUBSET high_fire / high_medium_fire
TRAINING_DATA_STAGE = "stage_c_knn"  # change to "stage_c" for median run
TRAINING_DATA_STAGE = os.environ.get("CHAMPION_TRAINING_STAGE", TRAINING_DATA_STAGE).strip()
if TRAINING_DATA_STAGE not in {"stage_c", "stage_c_knn"}:
    raise ValueError("TRAINING_DATA_STAGE must be 'stage_c_knn' or 'stage_c'")
_STAGE_PATHS = {
    "stage_c_knn": "/kaggle/input/datasets/lakshay654/california-wildfire-knn",
    "stage_c": "/kaggle/input/datasets/lakshay654/california-wildfire-median",
}
# Path follows the stage. Override only if your Kaggle slug differs.
DATA_DIRECTORY = os.environ.get("CHAMPION_DATA_DIR", _STAGE_PATHS[TRAINING_DATA_STAGE]).strip()
USE_PRECOMPUTED_KNN = TRAINING_DATA_STAGE == "stage_c_knn"
IMPUTATION_METHOD = "precomputed KNN" if USE_PRECOMPUTED_KNN else "training-only median"

# Fire-region cell subset (orthogonal to stage). Uses fire_analysis2.csv.
CELL_SUBSET = "all"  # all | high_fire | high_medium_fire
CELL_SUBSET = os.environ.get("CHAMPION_CELL_SUBSET", CELL_SUBSET).strip()
if CELL_SUBSET not in {"all", "high_fire", "high_medium_fire"}:
    raise ValueError("CELL_SUBSET must be 'all', 'high_fire', or 'high_medium_fire'")
CELL_SUBSET_CATEGORIES = {
    "all": None,
    "high_fire": ["High Outlier", "High"],
    "high_medium_fire": ["High Outlier", "High", "Medium"],
}
FIRE_REGION_CSV = os.environ.get(
    "CHAMPION_FIRE_REGION_CSV",
    "/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv",
).strip()  # used when CELL_SUBSET is high_fire / high_medium_fire

MODE = os.environ.get("CHAMPION_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("CHAMPION_MODE must be 'full' or 'smoke'")
USE_GPU = os.environ.get("CHAMPION_USE_GPU", "1").strip().lower() not in {"0", "false", "no"}
np.random.seed(SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
set_config(display="diagram")

def show(value: Any) -> None:
    try:
        from IPython.display import display
        display(value)
    except ImportError:
        print(value)

def show_image(path: Path) -> None:
    try:
        from IPython.display import Image, display
        display(Image(filename=str(path)))
    except ImportError:
        print(f"Saved image: {path}")

def candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    configured = DATA_DIRECTORY.strip() or os.environ.get("CHAMPION_DATA_DIR", "").strip()
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend([
        Path("/kaggle/input/datasets/lakshay654/california-wildfire-knn"),
        Path("/kaggle/input/datasets/lakshay654/california-wildfire-median"),
        Path("/kaggle/input/datasets/roushanks/dsai-datasets"),
        Path("/kaggle/input/dsai-datasets"),
    ])
    if Path("/kaggle/input").is_dir():
        candidates.extend(path.parent for path in Path("/kaggle/input").rglob("dataset_metadata.json"))
    current = Path.cwd().resolve()
    candidates.extend(base / "Archive" for base in (current, *current.parents))
    return list(dict.fromkeys(path.resolve() for path in candidates))

def read_metadata_stage(path: Path) -> str | None:
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("stage")
    except (OSError, json.JSONDecodeError):
        return None

def locate_data(expected_stage: str) -> dict[str, Path | str]:
    checked: list[str] = []
    for root in candidate_roots():
        stage_folder = "stage_c_knn" if expected_stage == "stage_c_knn" else "stage_c"
        layouts = [
            {
                "layout": "flat four-file", "root": root,
                "all": root / "all.parquet", "meta": root / "meta.json",
                "dataset_metadata": root / "dataset_metadata.json", "features": root / "feature_columns.json",
            },
            {
                "layout": "extracted local", "root": root / stage_folder,
                "all": root / stage_folder / "all.parquet", "meta": root / stage_folder / "meta.json",
                "dataset_metadata": root / stage_folder / "metadata" / "dataset_metadata.json",
                "features": root / stage_folder / "metadata" / "feature_columns.json",
            },
            {
                "layout": "legacy Stage C", "root": root,
                "all": root / "stage_c" / "all.parquet", "meta": root / "meta.json",
                "dataset_metadata": root / "stage_c" / "metadata" / "dataset_metadata.json",
                "features": root / "stage_c" / "metadata" / "feature_columns.json",
            },
        ]
        for layout in layouts:
            ready = all(Path(layout[key]).is_file() for key in ("all", "meta", "dataset_metadata", "features"))
            stage = read_metadata_stage(Path(layout["dataset_metadata"])) if ready else None
            checked.append(f"{layout['layout']}: {layout['root']} (stage={stage})")
            if ready and stage == expected_stage:
                return layout
    raise FileNotFoundError(
        f"Could not find a four-file dataset with stage={expected_stage}. Set DATA_DIRECTORY. Checked:\n"
        + "\n".join(f"  - {item}" for item in checked)
    )

DATA = locate_data(TRAINING_DATA_STAGE)
DATA_ROOT = Path(DATA["root"])
ALL_PARQUET = Path(DATA["all"])
META_JSON = Path(DATA["meta"])
DATASET_METADATA_JSON = Path(DATA["dataset_metadata"])
FEATURE_COLUMNS_JSON = Path(DATA["features"])
default_output = (
    Path(f"/kaggle/working/champion_training_outputs_{TRAINING_DATA_STAGE}_{CELL_SUBSET}")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "notebook_outputs" / f"champion_training_{TRAINING_DATA_STAGE}_{CELL_SUBSET}_{MODE}"
)
OUTPUT_DIR = Path(os.environ.get("CHAMPION_OUTPUT_DIR", str(default_output))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def reported_gpu() -> str | None:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            check=True, capture_output=True, text=True, timeout=10,
        )
        values = [line.strip() for line in result.stdout.splitlines() if line.strip()]
        return ", ".join(values) if values else None
    except (FileNotFoundError, subprocess.SubprocessError):
        return None

def lightgbm_device() -> tuple[str, dict[str, str]]:
    if not USE_GPU:
        return "cpu", {"cpu": "GPU disabled"}
    rng = np.random.default_rng(SEED)
    x = rng.normal(size=(2000, 20)).astype("float32")
    y = rng.integers(0, 2, size=2000, dtype="int8")
    failures: dict[str, str] = {}
    for device in ("cuda", "gpu"):
        try:
            dataset = lgb.Dataset(x, label=y)
            lgb.train(
                {"objective": "binary", "device_type": device, "verbosity": -1, "seed": SEED},
                dataset, num_boost_round=2,
            )
            return device, failures
        except Exception as error:
            failures[device] = str(error).splitlines()[-1][:400]
    return "cpu", failures

DEVICE, DEVICE_FAILURES = lightgbm_device()
GPU_NAME = reported_gpu()

print("=" * 72)
print("CHAMPION TRAINING CONFIGURATION")
print("=" * 72)
print(f"Training stage:    {TRAINING_DATA_STAGE}")
print(f"Cell subset:       {CELL_SUBSET}")
print(f"Imputation:        {IMPUTATION_METHOD}")
print(f"Mode:              {MODE}")
print(f"Input layout:      {DATA['layout']}")
print(f"Input directory:   {DATA_ROOT}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Reported GPU:      {GPU_NAME or 'not reported'}")
print(f"LightGBM device:   {DEVICE}")
if USE_GPU and DEVICE == "cpu":
    print("GPU requested but unavailable to this LightGBM build; using CPU safely.")
    for name, message in DEVICE_FAILURES.items():
        print(f"  {name}: {message}")
print("=" * 72)


## 2. Verify the selected four-file data2.0 dataset

The notebook resolves either `california-wildfire-knn` or `california-wildfire-median` from `TRAINING_DATA_STAGE`, then checks metadata stage, coverage, feature allowlist, causal timing, and—when KNN is selected—the exact train-only donor contract.


In [ ]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

archive_meta = read_json(META_JSON)
dataset_meta = read_json(DATASET_METADATA_JSON)
raw_features = read_json(FEATURE_COLUMNS_JSON)
parquet_file = pq.ParquetFile(ALL_PARQUET)
schema_columns = set(parquet_file.schema_arrow.names)
if dataset_meta.get("stage") != TRAINING_DATA_STAGE:
    raise ValueError("The selected metadata stage changed after discovery")
if dataset_meta.get("s5p_2021_mode") != "ready":
    raise ValueError("The data2.0 table must report S5P 2021 as ready")

missing_flag_column = None
if USE_PRECOMPUTED_KNN:
    missing_flag_column = "s2n_knn_imputed"
    if missing_flag_column not in schema_columns:
        raise ValueError("stage_c_knn must contain s2n_knn_imputed")
    knn_metadata = dataset_meta.get("knn", {})
    if (
        knn_metadata.get("n_neighbors") != 5
        or knn_metadata.get("weights") != "distance"
        or knn_metadata.get("donor_pool") != "train_years_<=_2022_and_s2n_available_eq_1"
    ):
        raise ValueError("The KNN metadata does not match the verified train-only data2.0 contract")

model_source_features = list(dict.fromkeys([
    *raw_features,
    *([missing_flag_column] if missing_flag_column else []),
]))
coverage_probe = pd.read_parquet(ALL_PARQUET, columns=["cell_id", "label_date", "y_fire"])
SOURCE_ROWS = int(len(coverage_probe))
SOURCE_CELLS = int(coverage_probe["cell_id"].nunique())
SOURCE_DAYS = int(pd.to_datetime(coverage_probe["label_date"]).nunique())
SOURCE_POSITIVES = int(coverage_probe["y_fire"].sum())
SOURCE_POSITIVE_RATE = SOURCE_POSITIVES / SOURCE_ROWS
del coverage_probe
gc.collect()
if SOURCE_ROWS != int(dataset_meta["n_rows"]) or SOURCE_POSITIVES != int(dataset_meta["n_pos"]):
    raise ValueError("Parquet coverage does not match dataset_metadata.json")

ID_COLUMNS = [
    "feature_end_date", "eo_asof_date", "label_date", "cell_id",
    "latitude", "longitude", "y_fire",
]
AGE_COLUMNS = ["s2n_lag_days", "s5n_lag_days"]
required_columns = set(ID_COLUMNS + AGE_COLUMNS + model_source_features)
missing_columns = sorted(required_columns - schema_columns)
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

input_files = [ALL_PARQUET, META_JSON, DATASET_METADATA_JSON, FEATURE_COLUMNS_JSON]
show(pd.DataFrame([
    {
        "file": path.name,
        "format": path.suffix.lstrip(".").upper(),
        "size_mb": path.stat().st_size / 1024**2,
        "purpose": (
            "complete cell-by-day table" if path == ALL_PARQUET
            else "compact archive metadata" if path == META_JSON
            else "stage, split, and preprocessing metadata" if path == DATASET_METADATA_JSON
            else "exact Stage C feature allowlist"
        ),
    }
    for path in input_files
]).round({"size_mb": 3}))

quantity_overview = pd.DataFrame([
    {"measure": "Rows", "quantity": SOURCE_ROWS},
    {"measure": "Columns on disk", "quantity": len(parquet_file.schema_arrow.names)},
    {"measure": "Stage C allowlisted features", "quantity": len(raw_features)},
    {"measure": "Additional KNN flag features", "quantity": int(USE_PRECOMPUTED_KNN)},
    {"measure": "Calendar days", "quantity": SOURCE_DAYS},
    {"measure": "California grid cells", "quantity": SOURCE_CELLS},
    {"measure": "Positive cell-days", "quantity": SOURCE_POSITIVES},
    {"measure": "Positive rate", "quantity": SOURCE_POSITIVE_RATE},
])
show(quantity_overview)
if USE_PRECOMPUTED_KNN:
    print(f"Verified KNN donor pool: {dataset_meta['knn']['donor_pool']}")
print(f"Verified coverage: {SOURCE_CELLS:,} cells × {SOURCE_DAYS:,} days = {SOURCE_ROWS:,} rows")

## 3. Load and apply stage-appropriate preprocessing

KNN input (`california-wildfire-knn`) is validated and passed through without a second imputation. Non-imputed Stage C (`california-wildfire-median`) marks unavailable S2 values as missing so sklearn learns medians from training only.

When `CELL_SUBSET` is `high_fire` or `high_medium_fire`, rows are filtered to cells listed in `firms-test/fire_analysis2.csv` before feature building.


In [ ]:
CONSTANT_FEATURES = {"s5n_s5p_aai_std", "s5n_s5p_co_std"}
base_features = [name for name in model_source_features if name not in CONSTANT_FEATURES]
load_columns = list(dict.fromkeys([*ID_COLUMNS, *AGE_COLUMNS, *model_source_features]))

def clean_source_table(frame: pd.DataFrame, raw_base_features: list[str]):
    """Apply only the preprocessing appropriate to the selected source stage."""
    frame = frame.copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        frame[name] = pd.to_datetime(frame[name]).dt.normalize()
    frame = frame.sort_values(["cell_id", "label_date"]).reset_index(drop=True)
    cleanup_counts: dict[str, int] = {}

    if USE_PRECOMPUTED_KNN:
        frame[raw_base_features] = frame[raw_base_features].apply(pd.to_numeric, errors="raise").astype("float32")
        flag_values = set(frame[missing_flag_column].dropna().unique().tolist())
        if not flag_values.issubset({0.0, 1.0}):
            raise ValueError(f"{missing_flag_column} must be binary; found {sorted(flag_values)}")
        if not np.isfinite(frame[raw_base_features].to_numpy(dtype="float32", copy=False)).all():
            raise ValueError("Precomputed KNN values contain NaN or infinity")
        flagged = frame[missing_flag_column].eq(1)
        if not frame.loc[flagged, "s2n_available"].eq(0).all():
            raise ValueError("Every KNN-imputed S2 row must retain s2n_available=0")
        cleanup_counts["knn_imputed_rows_flagged"] = int(flagged.sum())
    else:
        s2_invalid = frame["s2n_available"].ne(1)
        s2_values = [name for name in raw_base_features if name.startswith("s2n_") and name != "s2n_available"]
        frame.loc[s2_invalid, s2_values] = np.nan
        frame.loc[s2_invalid, "s2n_available"] = 0.0
        cleanup_counts["sentinel2_rows_marked_missing"] = int(s2_invalid.sum())
        s5_invalid = frame["s5n_available"].ne(1)
        s5_values = [name for name in raw_base_features if name.startswith("s5n_") and name != "s5n_available"]
        frame.loc[s5_invalid, s5_values] = 0.0
        frame.loc[s5_invalid, "s5n_available"] = 0.0
        cleanup_counts["sentinel5p_rows_zeroed"] = int(s5_invalid.sum())
        frame[raw_base_features] = frame[raw_base_features].astype("float32")

    for name in ("swvl1_mean", "swvl2_mean", "soil_moisture_index", "swvl1_mean_7d"):
        cleanup_counts[f"{name}_negative_rows_clipped"] = int(frame[name].lt(0).sum())
        frame[name] = frame[name].clip(lower=0)
    frame["year"] = frame["label_date"].dt.year.astype("int16")
    return frame, cleanup_counts

started = time.time()
def resolve_fire_region_csv() -> Path:
    configured = (FIRE_REGION_CSV or "").strip()
    candidates: list[Path] = []
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.append(Path("/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv"))
    candidates.append(Path.cwd() / "fire_analysis2.csv")
    # Notebook / Milestone 5 directory heuristics
    here = Path.cwd().resolve()
    candidates.extend([
        here / "Milestone 5" / "fire_analysis2.csv",
        here.parent / "Milestone 5" / "fire_analysis2.csv",
    ])
    for base in (here, *here.parents):
        candidates.append(base / "fire_analysis2.csv")
        candidates.append(base / "Milestone 5" / "fire_analysis2.csv")
    if Path("/kaggle/input").is_dir():
        candidates.extend(Path("/kaggle/input").rglob("fire_analysis2.csv"))
    seen: set[Path] = set()
    for path in candidates:
        path = path.resolve()
        if path in seen:
            continue
        seen.add(path)
        if path.is_file():
            return path
    raise FileNotFoundError(
        "fire_analysis2.csv not found for CELL_SUBSET="
        f"{CELL_SUBSET}. Set FIRE_REGION_CSV or attach the CSV under /kaggle/input."
    )


def cells_for_subset(csv_path: Path, categories: list[str], available_cells: list) -> list:
    frame = pd.read_csv(csv_path)
    required = {"cell_id", "fire_region_category"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"fire_analysis2.csv missing columns: {sorted(missing)}")
    frame["cell_id"] = frame["cell_id"].astype(str)
    available = {str(cell) for cell in available_cells}
    counts = frame["fire_region_category"].value_counts(dropna=False).to_dict()
    print("fire_analysis2.csv category counts:", counts)
    allow = set(
        frame.loc[frame["fire_region_category"].isin(categories), "cell_id"].astype(str)
    )
    selected = [cell for cell in available_cells if str(cell) in allow]
    if not selected:
        raise ValueError(
            f"No Stage C cells matched CELL_SUBSET={CELL_SUBSET} categories={categories}"
        )
    unmatched_csv = sorted(allow - available)
    print(
        f"CELL_SUBSET={CELL_SUBSET}: categories={categories} "
        f"selected_cells={len(selected)} "
        f"csv_only_not_in_stage_c={len(unmatched_csv)}"
    )
    return selected


cell_table = pd.read_parquet(ALL_PARQUET, columns=["cell_id"])
archive_cells = sorted(cell_table["cell_id"].unique().tolist())
selected_cells = list(archive_cells)
FIRE_REGION_CSV_PATH = None
if MODE == "smoke":
    positions = np.linspace(0, len(selected_cells) - 1, min(96, len(selected_cells)), dtype=int)
    selected_cells = list(dict.fromkeys(selected_cells[position] for position in positions))
if CELL_SUBSET != "all":
    FIRE_REGION_CSV_PATH = resolve_fire_region_csv()
    selected_cells = cells_for_subset(
        FIRE_REGION_CSV_PATH,
        CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        selected_cells,
    )
del cell_table
gc.collect()

SELECTED_TRAINING_CELLS = list(selected_cells)
CELL_SUBSET_INFO = {
    "cell_subset": CELL_SUBSET,
    "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
    "n_cells_selected": len(SELECTED_TRAINING_CELLS),
    "n_archive_cells": len(archive_cells),
    "fire_region_csv": str(FIRE_REGION_CSV_PATH) if FIRE_REGION_CSV_PATH else None,
}

if len(SELECTED_TRAINING_CELLS) == len(archive_cells) and CELL_SUBSET == "all" and MODE != "smoke":
    data = pd.read_parquet(ALL_PARQUET, columns=load_columns)
else:
    data = pd.read_parquet(
        ALL_PARQUET,
        columns=load_columns,
        filters=[("cell_id", "in", SELECTED_TRAINING_CELLS)],
    )

data, cleanup = clean_source_table(data, base_features)
cells = int(data["cell_id"].nunique())
days = int(data["label_date"].nunique())
assert len(data) == cells * days
assert data.duplicated(["cell_id", "label_date"]).sum() == 0
assert sorted(data["y_fire"].unique().tolist()) == [0, 1]
assert (data["label_date"] - data["eo_asof_date"]).dt.days.eq(1).all()
assert (data["eo_asof_date"] - data["feature_end_date"]).dt.days.eq(5).all()
if cells != len(SELECTED_TRAINING_CELLS):
    raise ValueError(
        f"Loaded cells ({cells}) do not match selected subset ({len(SELECTED_TRAINING_CELLS)})"
    )
print(
    f"Loaded subset: cells={cells:,} days={days:,} rows={len(data):,} "
    f"positives={int(data['y_fire'].sum()):,} CELL_SUBSET={CELL_SUBSET}"
)
if USE_PRECOMPUTED_KNN and MODE == "full" and CELL_SUBSET == "all":
    if cleanup["knn_imputed_rows_flagged"] != int(dataset_meta["n_s2_knn_imputed"]):
        raise ValueError("Full KNN flag count does not match metadata")

yearly_quantity = data.groupby("year").agg(
    rows=("y_fire", "size"), positives=("y_fire", "sum"), days=("label_date", "nunique"),
)
yearly_quantity["positive_rate"] = yearly_quantity["positives"] / yearly_quantity["rows"]
yearly_quantity_display = yearly_quantity.copy()
yearly_quantity_display["positive_rate"] = yearly_quantity_display["positive_rate"].map(lambda value: f"{value:.3%}")
show(yearly_quantity_display)
print(json.dumps(cleanup, indent=2))
print(f"Loaded {len(data):,} rows in {time.time() - started:.1f}s")

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(9, 4.5))
axis.bar(yearly_quantity.index.astype(str), yearly_quantity["positives"], color="#c9472c")
axis.set(title="Positive wildfire cell-days by year", xlabel="Label year", ylabel="Positive rows")
for position, value in enumerate(yearly_quantity["positives"]):
    axis.text(position, value, f"{int(value):,}", ha="center", va="bottom", fontsize=9)
figure.tight_layout()
quantity_plot = plots_dir / "data_quantity_by_year.png"
figure.savefig(quantity_plot, dpi=150, bbox_inches="tight")
plt.close(figure)
show_image(quantity_plot)


## 4. Build the champion feature table

This is the only substantial custom transformation block. It is required because the four input files contain the source measurements but not the champion's temporal and spatial context features. Scikit-learn cannot infer these domain rules automatically.

The block creates:

- 4 calendar-cycle features;
- 23 weather-history and interaction features;
- 10 strictly lagged fire-history features ending at `D-1`;
- 14 wind-aware neighbor and dryness features.

The implementation cell is collapsed by default to keep the main notebook readable. Expand it only when reviewing the feature formulas.


In [ ]:
def rolling_matrix(values: np.ndarray, window: int, operation: str) -> np.ndarray:
    rolling = pd.DataFrame(values.T).rolling(window=window, min_periods=1)
    return getattr(rolling, operation)().to_numpy(dtype="float32").T

def simple_neighbors(frame: pd.DataFrame, cells: int, days: int) -> list[np.ndarray]:
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    result = []
    for latitude, longitude in coordinates:
        delta_lat = np.abs(coordinates[:, 0] - latitude)
        delta_lon = np.abs(coordinates[:, 1] - longitude)
        mask = (
            (delta_lat <= 0.251)
            & (delta_lon <= 0.251)
            & ~((delta_lat < 1e-9) & (delta_lon < 1e-9))
        )
        result.append(np.flatnonzero(mask))
    return result

def sum_neighbors(values: np.ndarray, neighbors: list[np.ndarray]) -> np.ndarray:
    output = np.zeros_like(values, dtype="float32")
    for index, adjacent in enumerate(neighbors):
        if adjacent.size:
            output[index] = values[adjacent].sum(axis=0)
    return output

def directional_geometry(coordinates: np.ndarray):
    result = []
    for latitude, longitude in coordinates:
        delta_lat = latitude - coordinates[:, 0]
        delta_lon = (longitude - coordinates[:, 1]) * np.cos(np.deg2rad(latitude))
        distance = np.sqrt(delta_lat**2 + delta_lon**2)
        adjacent = np.flatnonzero((distance > 1e-9) & (distance <= 0.36))
        result.append((
            adjacent,
            (delta_lon[adjacent] / distance[adjacent]).astype("float32"),
            (delta_lat[adjacent] / distance[adjacent]).astype("float32"),
            (1.0 / (distance[adjacent] + 0.05)).astype("float32"),
        ))
    return result

def directional_counts(fire: np.ndarray, wind_sin: np.ndarray, wind_cos: np.ndarray, geometry):
    cells, days = fire.shape
    upwind = np.zeros((cells, days), dtype="float32")
    downwind = np.zeros((cells, days), dtype="float32")
    crosswind = np.zeros((cells, days), dtype="float32")
    distance_weighted = np.zeros((cells, days), dtype="float32")
    wind_east, wind_north = -wind_sin, -wind_cos
    for index, (adjacent, east, north, inverse_distance) in enumerate(geometry):
        if not adjacent.size:
            continue
        neighbor_fire = fire[adjacent]
        alignment = wind_east[index][None, :] * east[:, None] + wind_north[index][None, :] * north[:, None]
        upwind[index] = (neighbor_fire * np.maximum(alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        downwind[index] = (neighbor_fire * np.maximum(-alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        crosswind[index] = (
            neighbor_fire * np.sqrt(np.maximum(1.0 - alignment**2, 0)) * inverse_distance[:, None]
        ).sum(axis=0)
        distance_weighted[index] = (neighbor_fire * inverse_distance[:, None]).sum(axis=0)
    return upwind, downwind, crosswind, distance_weighted

def build_features(frame: pd.DataFrame, raw_base_features: list[str]):
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())

    # Calendar cycles.
    day_of_year = frame["eo_asof_date"].dt.dayofyear.to_numpy(dtype="float32")
    month = frame["eo_asof_date"].dt.month.to_numpy(dtype="float32")
    calendar = pd.DataFrame({
        "day_of_year_sin": np.sin(2 * np.pi * day_of_year / 365.25),
        "day_of_year_cos": np.cos(2 * np.pi * day_of_year / 365.25),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
    }, index=frame.index).astype("float32")
    frame = pd.concat([frame, calendar], axis=1)

    # Weather physics and history ending at D-5.
    temperature_c = frame["t2m_mean"].to_numpy(dtype="float64") - 273.15
    dewpoint_c = frame["d2m_mean"].to_numpy(dtype="float64") - 273.15
    saturation = 0.6108 * np.exp(17.27 * temperature_c / np.maximum(temperature_c + 237.3, 1e-6))
    actual = 0.6108 * np.exp(17.27 * dewpoint_c / np.maximum(dewpoint_c + 237.3, 1e-6))
    vpd = np.maximum(saturation - actual, 0).astype("float32")
    weather = {
        "vpd_kpa": vpd,
        "vpd_wind_interaction": vpd * frame["wind_speed_mean"].to_numpy(dtype="float32"),
        "vpd_soil_deficit_interaction": vpd * (1 - np.clip(frame["soil_moisture_index"], 0, 1)),
        "heat_soil_deficit_interaction": (
            np.maximum(frame["t2m_max"].to_numpy(dtype="float32") - 273.15, 0)
            * (1 - np.clip(frame["swvl1_mean"], 0, 1))
        ),
        "wind_gust_ratio": frame["i10fg_max"].to_numpy(dtype="float32")
        / (frame["wind_speed_mean"].to_numpy(dtype="float32") + 0.1),
    }
    rolling_specs = {
        "t2m_max": ("max",), "rh_mean": ("min",), "tp_sum_mm": ("sum",),
        "wind_speed_mean": ("max",), "i10fg_max": ("max",),
        "swvl1_mean": ("mean",), "vpd_kpa": ("max", "mean"),
    }
    arrays = {
        name: (weather[name] if name in weather else frame[name].to_numpy(dtype="float32")).reshape(cells, days)
        for name in rolling_specs
    }
    for name, operations in rolling_specs.items():
        for window in (14, 30):
            for operation in operations:
                weather[f"{name}_{operation}_{window}d"] = rolling_matrix(
                    arrays[name], window, operation
                ).reshape(-1)
    temperature = frame["t2m_max"].to_numpy(dtype="float32").reshape(cells, days)
    soil = frame["swvl1_mean"].to_numpy(dtype="float32").reshape(cells, days)
    weather["t2m_max_anomaly_30d"] = (temperature - rolling_matrix(temperature, 30, "mean")).reshape(-1)
    weather["swvl1_anomaly_30d"] = (soil - rolling_matrix(soil, 30, "mean")).reshape(-1)
    weather_frame = pd.DataFrame(weather, index=frame.index).astype("float32")
    frame = pd.concat([frame, weather_frame], axis=1)

    # Fire history ending at D-1 (two target rows behind label day D+1).
    target = frame["y_fire"].to_numpy(dtype="float32").reshape(cells, days)
    lag2 = np.zeros_like(target, dtype="float32")
    lag2[:, 2:] = target[:, :-2]
    history7 = rolling_matrix(lag2, 7, "sum")
    history30 = rolling_matrix(lag2, 30, "sum")
    neighbors = simple_neighbors(frame, cells, days)
    neighbor_lag2 = sum_neighbors(lag2, neighbors)
    neighbor_7d = sum_neighbors(history7, neighbors)
    last_positive = np.full(cells, -10_000, dtype="int32")
    days_since = np.full_like(target, 365, dtype="float32")
    for day in range(days):
        positive = lag2[:, day] > 0
        last_positive[positive] = day
        seen = last_positive > -10_000
        days_since[seen, day] = np.minimum(day - last_positive[seen], 365)
    observed = np.maximum(np.arange(days, dtype="float32") - 1, 0)
    expanding_rate = (np.cumsum(lag2, axis=1, dtype="float32") + 1.0) / (observed[None, :] + 100.0)
    fire = {
        "fire_cell_lag2": lag2.reshape(-1),
        "fire_cell_count_7d_lag2": history7.reshape(-1),
        "fire_cell_count_30d_lag2": history30.reshape(-1),
        "fire_cell_any_7d_lag2": (history7 > 0).astype("float32").reshape(-1),
        "fire_cell_days_since_lag2": days_since.reshape(-1),
        "fire_cell_expanding_rate_lag2": expanding_rate.reshape(-1),
        "fire_neighbor_count_lag2": neighbor_lag2.reshape(-1),
        "fire_neighbor_count_7d_lag2": neighbor_7d.reshape(-1),
        "fire_neighbor_any_7d_lag2": (neighbor_7d > 0).astype("float32").reshape(-1),
        "fire_statewide_cells_7d_lag2": np.tile(history7.sum(axis=0), cells).astype("float32"),
    }
    fire_frame = pd.DataFrame(fire, index=frame.index).astype("float32")
    frame = pd.concat([frame, fire_frame], axis=1)

    # Wind-aware neighboring fire context.
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    geometry = directional_geometry(coordinates)
    wind_sin = frame["wind_dir_sin"].to_numpy(dtype="float32").reshape(cells, days)
    wind_cos = frame["wind_dir_cos"].to_numpy(dtype="float32").reshape(cells, days)
    upwind_lag2, _, _, distance_lag2 = directional_counts(lag2, wind_sin, wind_cos, geometry)
    upwind_7d, downwind_7d, crosswind_7d, distance_7d = directional_counts(
        history7, wind_sin, wind_cos, geometry
    )
    wind = frame["wind_speed_mean"].to_numpy(dtype="float32")
    vpd = frame["vpd_kpa"].to_numpy(dtype="float32")
    soil_deficit = 1 - np.clip(frame["soil_moisture_index"].to_numpy(dtype="float32"), 0, 1)
    vegetation = np.clip(
        frame["cvh_mean"].to_numpy(dtype="float32") + frame["cvl_mean"].to_numpy(dtype="float32"), 0, 1
    )
    recent_context = np.maximum(
        frame["fire_cell_any_7d_lag2"].to_numpy(dtype="float32"),
        frame["fire_neighbor_any_7d_lag2"].to_numpy(dtype="float32"),
    )
    directional = {
        "fire_upwind_count_lag2": upwind_lag2.reshape(-1),
        "fire_upwind_count_7d_lag2": upwind_7d.reshape(-1),
        "fire_downwind_count_7d_lag2": downwind_7d.reshape(-1),
        "fire_crosswind_count_7d_lag2": crosswind_7d.reshape(-1),
        "fire_distance_weighted_count_lag2": distance_lag2.reshape(-1),
        "fire_distance_weighted_count_7d_lag2": distance_7d.reshape(-1),
        "fire_wind_spread_potential_lag2": upwind_lag2.reshape(-1) * wind,
        "fire_wind_spread_potential_7d_lag2": upwind_7d.reshape(-1) * wind,
        "fire_context_vpd_interaction": frame["fire_neighbor_count_7d_lag2"].to_numpy(dtype="float32") * vpd,
        "fire_context_dry_windy_interaction": recent_context * vpd * wind * soil_deficit,
        "ignition_dry_windy_index": vpd * wind * soil_deficit,
        "fuel_dryness_index": vpd * soil_deficit * vegetation,
        "vpd_short_long_trend": frame["vpd_kpa_mean_14d"] - frame["vpd_kpa_mean_30d"],
        "recent_fire_context": recent_context,
    }
    directional_frame = pd.DataFrame(directional, index=frame.index).astype("float32")
    frame = pd.concat([frame, directional_frame], axis=1)

    # The locked contract excludes a redundant source-availability flag.
    selected_base = [name for name in raw_base_features if name != "s5n_available"]
    feature_columns = list(dict.fromkeys([
        *selected_base,
        "latitude", "longitude",
        *calendar.columns,
        *weather_frame.columns,
        *fire_frame.columns,
        *directional_frame.columns,
    ]))
    groups = {
        "source_after_constant_removal": len(raw_base_features),
        "source_used_by_model": len(selected_base),
        "geographic": 2,
        "calendar": len(calendar.columns),
        "weather_and_interactions": len(weather_frame.columns),
        "causal_fire_history": len(fire_frame.columns),
        "wind_and_context": len(directional_frame.columns),
        "total": len(feature_columns),
    }
    return frame, feature_columns, groups


In [ ]:
started = time.time()
data, feature_columns, feature_groups = build_features(data, base_features)
EXPECTED_FEATURE_COUNT = 114 if USE_PRECOMPUTED_KNN else 113
if len(feature_columns) != EXPECTED_FEATURE_COUNT:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COUNT} champion features, found {len(feature_columns)}")
values = data[feature_columns].to_numpy(dtype="float32", copy=False)
if np.isinf(values).any():
    raise ValueError("An infinite engineered feature was found")
if USE_PRECOMPUTED_KNN and np.isnan(values).any():
    raise ValueError("KNN feature table contains NaN")
missing_feature_values = int(np.isnan(values).sum())

show(pd.DataFrame([
    {"feature_group": name.replace("_", " ").title(), "count": count}
    for name, count in feature_groups.items()
]))
print(f"Created the {EXPECTED_FEATURE_COUNT}-feature table in {time.time() - started:.1f}s")
print(f"Values awaiting pipeline imputation: {missing_feature_values:,}")

## 5. Chronological model splits

The source contract is followed exactly and no random row split is used:

- **Model fitting:** 2019–2022
- **Validation:** 2023
- **Probability calibration:** 2024
- **Final test:** 2025

The source `val` period is separated internally so validation and calibration do not reuse the same rows. Feature engineering happens before masking only to preserve causal rolling history across year boundaries; every temporal feature uses information strictly earlier than its target.

In [ ]:
training = data.loc[data["year"].le(2022)].copy()
validation = data.loc[data["year"].eq(2023)].copy()
calibration = data.loc[data["year"].eq(2024)].copy()
test = data.loc[data["year"].eq(2025)].copy()
del data, values
gc.collect()

split_quantity = pd.DataFrame([
    {
        "split": name,
        "label_years": f"{part['year'].min()}–{part['year'].max()}" if part["year"].nunique() > 1 else str(part["year"].iloc[0]),
        "rows": len(part),
        "positives": int(part["y_fire"].sum()),
        "positive_rate": float(part["y_fire"].mean()),
        "purpose": purpose,
    }
    for name, part, purpose in [
        ("training", training, "fit classifier, ranker, and preprocessing"),
        ("validation", validation, "check the fixed champion configuration"),
        ("calibration", calibration, "fit probability calibrator only"),
        ("test", test, "final descriptive evaluation"),
    ]
])
split_quantity_display = split_quantity.copy()
split_quantity_display["rows"] = split_quantity_display["rows"].map(lambda value: f"{value:,}")
split_quantity_display["positives"] = split_quantity_display["positives"].map(lambda value: f"{value:,}")
split_quantity_display["positive_rate"] = split_quantity_display["positive_rate"].map(lambda value: f"{value:.3%}")
show(split_quantity_display)

## 6. Define the two sklearn training pipelines

The diagrams show either an identity pass-through for precomputed KNN values or a training-only median imputer, followed by the fixed LightGBM classifier or ranker.

In [ ]:
def make_preprocessor():
    if USE_PRECOMPUTED_KNN:
        return FunctionTransformer(validate=False, feature_names_out="one-to-one")
    return SimpleImputer(strategy="median")

classifier_pipeline = Pipeline([
    ("feature_preprocessor", make_preprocessor()),
    ("fire_probability_model", lgb.LGBMClassifier(
        objective="binary", n_estimators=248, learning_rate=0.025,
        num_leaves=31, min_child_samples=75, colsample_bytree=0.90,
        subsample=0.90, subsample_freq=1, reg_alpha=0.2, reg_lambda=3.0,
        scale_pos_weight=1.0, random_state=SEED, n_jobs=-1, verbosity=-1,
        feature_pre_filter=False, device_type=DEVICE,
    )),
])

ranker_pipeline = Pipeline([
    ("feature_preprocessor", make_preprocessor()),
    ("daily_priority_model", lgb.LGBMRanker(
        objective="lambdarank", n_estimators=221, learning_rate=0.03,
        num_leaves=63, min_child_samples=100, colsample_bytree=0.85,
        subsample=0.85, subsample_freq=1, reg_lambda=8.0,
        lambdarank_truncation_level=100, label_gain=[0, 1],
        random_state=SEED, n_jobs=-1, verbosity=-1,
        feature_pre_filter=False, device_type=DEVICE,
    )),
])

print("Classifier pipeline")
show(classifier_pipeline)
print("Ranker pipeline")
show(ranker_pipeline)

## 7. Fit the champion model

The classifier and ranker, including their preprocessing, are fitted only on 2019–2022. The locked configuration is reported on 2023, the probability calibrator is fitted only on 2024, and 2025 remains untouched until final evaluation.

In [ ]:
started = time.time()
classifier_pipeline.fit(training[feature_columns], training["y_fire"])
classifier_seconds = time.time() - started

rank_training = training.sort_values(["label_date", "cell_id"])
daily_group_sizes = rank_training.groupby("label_date", sort=False).size().to_numpy(dtype="int32")
started = time.time()
ranker_pipeline.fit(
    rank_training[feature_columns],
    rank_training["y_fire"],
    daily_priority_model__group=daily_group_sizes,
)
ranker_seconds = time.time() - started

# The fixed configuration is checked on 2023 without changing any fitted component.
raw_validation_probability = classifier_pipeline.predict_proba(validation[feature_columns])[:, 1]
validation_metrics = {
    "rows": len(validation),
    "positives": int(validation["y_fire"].sum()),
    "pr_auc": float(average_precision_score(validation["y_fire"], raw_validation_probability)),
    "roc_auc": float(roc_auc_score(validation["y_fire"], raw_validation_probability)),
}
show(pd.DataFrame([validation_metrics]).round({"pr_auc": 6, "roc_auc": 6}))

# Only 2024 fits the probability calibrator.
raw_calibration_probability = classifier_pipeline.predict_proba(
    calibration[feature_columns]
)[:, 1]
clipped = np.clip(raw_calibration_probability, 1e-7, 1 - 1e-7)
calibration_logit = np.log(clipped / (1 - clipped)).reshape(-1, 1)
probability_calibrator = LogisticRegression(
    C=1e6, solver="lbfgs", max_iter=1000, random_state=SEED
).fit(calibration_logit, calibration["y_fire"])

training_summary = pd.DataFrame([
    {"component": "classifier pipeline", "device": DEVICE, "fit_seconds": classifier_seconds, "training_rows": len(training)},
    {"component": "ranker pipeline", "device": DEVICE, "fit_seconds": ranker_seconds, "training_rows": len(rank_training)},
    {"component": "probability calibrator", "device": "cpu", "fit_seconds": math.nan, "training_rows": len(calibration)},
])
training_summary_display = training_summary.copy()
training_summary_display["fit_seconds"] = training_summary_display["fit_seconds"].map(
    lambda value: "—" if pd.isna(value) else f"{value:.1f}"
)
training_summary_display["training_rows"] = training_summary_display["training_rows"].map(lambda value: f"{value:,}")
show(training_summary_display)

## 8. Predict and evaluate 2025

The calibrated probability and the alert ranking have different meanings:

- `p_fire` is the estimated next-day probability.
- `alert_score` is a 50/50 blend of the classifier and ranker percentiles within each date.

PR-AUC and daily Recall@25 are emphasized because positive wildfire rows are rare.


In [ ]:
def within_day_percentile(score: np.ndarray, dates: pd.Series) -> np.ndarray:
    table = pd.DataFrame({"date": pd.to_datetime(dates).to_numpy(), "score": score, "position": np.arange(len(score))})
    table["percentile"] = table.groupby("date", sort=False)["score"].rank(method="average", pct=True)
    return table.sort_values("position")["percentile"].to_numpy(dtype=float)

def top_k_metrics(frame: pd.DataFrame, k: int, score_column: str) -> dict[str, float | int]:
    alerts = (
        frame.sort_values(["label_date", score_column], ascending=[True, False])
        .groupby("label_date", group_keys=False)
        .head(k)
    )
    captured = int(alerts["y_fire"].sum())
    positives = int(frame["y_fire"].sum())
    days = int(frame["label_date"].nunique())
    return {
        "alerts": len(alerts),
        "precision": captured / len(alerts),
        "recall": captured / positives,
        "false_alerts_per_day": (len(alerts) - captured) / days,
    }

raw_probability = classifier_pipeline.predict_proba(test[feature_columns])[:, 1]
raw_clipped = np.clip(raw_probability, 1e-7, 1 - 1e-7)
calibrated_probability = probability_calibrator.predict_proba(
    np.log(raw_clipped / (1 - raw_clipped)).reshape(-1, 1)
)[:, 1]

rank_test = test.sort_values(["label_date", "cell_id"])
rank_score_sorted = ranker_pipeline.predict(rank_test[feature_columns])
rank_predictions = pd.Series(rank_score_sorted, index=rank_test.index).reindex(test.index).to_numpy(dtype=float)
classifier_percentile = within_day_percentile(raw_probability, test["label_date"])
ranker_percentile = within_day_percentile(rank_predictions, test["label_date"])
alert_score = 0.50 * classifier_percentile + 0.50 * ranker_percentile

scored = test[["feature_end_date", "eo_asof_date", "label_date", "cell_id", "latitude", "longitude", "y_fire"]].copy()
scored["p_fire_raw"] = raw_probability.astype("float32")
scored["p_fire"] = calibrated_probability.astype("float32")
scored["rank_score"] = rank_predictions.astype("float32")
scored["alert_score"] = alert_score.astype("float32")

y_test = scored["y_fire"].to_numpy(dtype="int8")
p_test = scored["p_fire"].to_numpy(dtype=float)
alert_at_25 = top_k_metrics(scored, 25, "alert_score")
alert_at_50 = top_k_metrics(scored, 50, "alert_score")
metrics = {
    "rows": len(scored),
    "positives": int(y_test.sum()),
    "prevalence": float(y_test.mean()),
    "pr_auc": float(average_precision_score(y_test, p_test)),
    "roc_auc": float(roc_auc_score(y_test, p_test)),
    "brier": float(brier_score_loss(y_test, p_test)),
    "log_loss": float(log_loss(y_test, p_test, labels=[0, 1])),
    "recall_at_25": alert_at_25["recall"],
    "precision_at_25": alert_at_25["precision"],
    "false_alerts_per_day_at_25": alert_at_25["false_alerts_per_day"],
    "recall_at_50": alert_at_50["recall"],
}
metrics_display = pd.DataFrame([metrics])
for name in ("prevalence", "recall_at_25", "precision_at_25", "recall_at_50"):
    metrics_display[name] = metrics_display[name].map(lambda value: f"{value:.3%}")
for name in ("pr_auc", "roc_auc", "brier", "log_loss"):
    metrics_display[name] = metrics_display[name].map(lambda value: f"{value:.6f}")
metrics_display["false_alerts_per_day_at_25"] = metrics_display["false_alerts_per_day_at_25"].map(
    lambda value: f"{value:.2f}"
)
show(metrics_display)


In [ ]:
observed, predicted = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
precision, recall, _ = precision_recall_curve(y_test, p_test)
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(predicted, observed, marker="o", label="champion")
axes[0].plot([0, 1], [0, 1], "--", color="black", label="ideal")
axes[0].set(title="Probability reliability", xlabel="Predicted probability", ylabel="Observed rate")
axes[0].legend()
axes[1].plot(recall, precision, color="#c9472c")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label="prevalence")
axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
axes[1].legend()
figure.tight_layout()
calibration_plot = plots_dir / "calibration_and_precision_recall.png"
figure.savefig(calibration_plot, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(calibration_plot)

peak_day = scored.groupby("label_date")["p_fire"].max().idxmax()
day = scored.loc[scored["label_date"].eq(peak_day)]
figure, axis = plt.subplots(figsize=(7, 8))
points = axis.scatter(
    day["longitude"], day["latitude"], c=day["p_fire"], cmap="YlOrRd",
    vmin=0, vmax=max(float(day["p_fire"].quantile(0.99)), 1e-5), s=28,
)
positives = day.loc[day["y_fire"].eq(1)]
if len(positives):
    axis.scatter(positives["longitude"], positives["latitude"], marker="x", color="black", s=38, label="observed positive")
    axis.legend()
axis.set(title=f"Next-day risk — {pd.Timestamp(peak_day).date()}", xlabel="Longitude", ylabel="Latitude")
figure.colorbar(points, ax=axis, label="Calibrated probability")
figure.tight_layout()
risk_map = plots_dir / "peak_day_risk_map.png"
figure.savefig(risk_map, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(risk_map)


## 9. Explain the classifier

The ranker determines relative daily ordering, while the classifier supplies probability. The explanation below therefore focuses on the classifier pipeline. LightGBM's native TreeSHAP contributions are used; no additional SHAP package is required.


In [ ]:
explain_dir = OUTPUT_DIR / "explainability"
explain_dir.mkdir(parents=True, exist_ok=True)
sample = test.sample(min(2000, len(test)), random_state=SEED)
feature_preprocessor = classifier_pipeline.named_steps["feature_preprocessor"]
classifier_model = classifier_pipeline.named_steps["fire_probability_model"]
matrix = feature_preprocessor.transform(sample[feature_columns])
contributions = classifier_model.booster_.predict(matrix, pred_contrib=True)
shap_values = np.asarray(contributions)[:, :-1]
importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_absolute_contribution": np.abs(shap_values).mean(axis=0),
    "gain_importance": classifier_model.booster_.feature_importance(importance_type="gain"),
}).sort_values("mean_absolute_contribution", ascending=False)
importance.to_csv(explain_dir / "feature_explanations.csv", index=False)
show(importance.head(20))

top = importance.head(20).sort_values("mean_absolute_contribution")
figure, axis = plt.subplots(figsize=(9, 7))
axis.barh(top["feature"], top["mean_absolute_contribution"], color="#c9472c")
axis.set(title="Most influential classifier features", xlabel="Mean absolute TreeSHAP contribution")
figure.tight_layout()
importance_plot = explain_dir / "feature_explanations.png"
figure.savefig(importance_plot, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(importance_plot)

## 10. Export the trained artifact and weights

Writes under `/kaggle/working/champion_training_outputs_<stage>_<subset>/` (or the local notebook output folder). The inference notebook loads `models/champion_model.joblib`. The two text files expose the underlying LightGBM weights, while the joblib artifact remains authoritative because it also contains preprocessing, probability calibration, and the selected cell list.


In [ ]:
def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (Path, pd.Timestamp, datetime)):
        return str(value)
    return value

def write_json(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True) + "\n", encoding="utf-8")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

models_dir = OUTPUT_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "champion_model.joblib"
classifier_weights_path = models_dir / "classifier_weights.txt"
ranker_weights_path = models_dir / "ranker_weights.txt"

model_artifact = {
    "artifact_name": "champion",
    "source_stage": TRAINING_DATA_STAGE,
    "cell_subset": CELL_SUBSET,
    "imputation_method": IMPUTATION_METHOD,
    "missing_flag_column": missing_flag_column,
    "classifier_pipeline": classifier_pipeline,
    "ranker_pipeline": ranker_pipeline,
    "probability_calibrator": probability_calibrator,
    "feature_columns": feature_columns,
    "raw_feature_columns": model_source_features,
    "base_features": base_features,
    "source_columns": load_columns,
    "classifier_weight": 0.50,
    "ranker_weight": 0.50,
    "data_contract": {
        "label_offset_from_eo_asof_days": 1,
        "eo_asof_offset_from_feature_end_days": 5,
        "expected_grid_cells": int(test["cell_id"].nunique()),
        "archive_source_cells": SOURCE_CELLS,
        "cell_subset": CELL_SUBSET,
        "cell_subset_categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        "selected_cell_ids": SELECTED_TRAINING_CELLS,
        "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
        "archive_last_label_date": str(test["label_date"].max().date()),
        "training_years": [2019, 2020, 2021, 2022],
        "validation_year": 2023,
        "calibration_year": 2024,
        "test_year": 2025,
    },
}
cell_subset_path = models_dir / "selected_cells.json"
write_json({
    "cell_subset": CELL_SUBSET,
    "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
    "n_cells_selected": len(SELECTED_TRAINING_CELLS),
    "cell_ids": SELECTED_TRAINING_CELLS,
    "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
}, cell_subset_path)

joblib.dump(model_artifact, model_path)
classifier_pipeline.named_steps["fire_probability_model"].booster_.save_model(str(classifier_weights_path))
ranker_pipeline.named_steps["daily_priority_model"].booster_.save_model(str(ranker_weights_path))

predictions_path = OUTPUT_DIR / "test_predictions.parquet"
scored.to_parquet(predictions_path, index=False)
feature_contract_path = OUTPUT_DIR / "feature_contract.json"
write_json({
    "source_stage": TRAINING_DATA_STAGE,
    "imputation_method": IMPUTATION_METHOD,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns,
    "feature_groups": feature_groups,
}, feature_contract_path)
metrics_path = OUTPUT_DIR / "metrics.json"
write_json({
    "model": "champion",
    "source_stage": TRAINING_DATA_STAGE,
    "imputation_method": IMPUTATION_METHOD,
    "cell_subset": CELL_SUBSET,
    "cell_subset_info": {
        "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        "n_cells_selected": len(SELECTED_TRAINING_CELLS),
        "n_archive_cells": SOURCE_CELLS,
        "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
    },
    "run_mode": MODE,
    "device": DEVICE,
    "reported_gpu": GPU_NAME,
    "data_quantity": {
        "source_rows": SOURCE_ROWS, "source_cells": SOURCE_CELLS,
        "source_days": SOURCE_DAYS, "source_positives": SOURCE_POSITIVES,
        "splits": split_quantity.to_dict(orient="records"),
    },
    "architecture": {
        "features": len(feature_columns),
        "classifier_pipeline": [IMPUTATION_METHOD, "LightGBM classifier"],
        "ranker_pipeline": [IMPUTATION_METHOD, "LightGBM ranker"],
        "daily_blend": {"classifier_percentile": 0.50, "ranker_percentile": 0.50},
    },
    "validation_2023_descriptive": validation_metrics,
    "test_2025_descriptive": metrics,
}, metrics_path)

# The artifact used by the inference notebook must reproduce both model branches exactly.
loaded = joblib.load(model_path)
verification = test.sort_values(["label_date", "cell_id"]).head(min(1000, len(test)))
original_probability = classifier_pipeline.predict_proba(verification[feature_columns])[:, 1]
loaded_probability = loaded["classifier_pipeline"].predict_proba(verification[feature_columns])[:, 1]
original_rank = ranker_pipeline.predict(verification[feature_columns])
loaded_rank = loaded["ranker_pipeline"].predict(verification[feature_columns])
probability_difference = float(np.max(np.abs(original_probability - loaded_probability)))
rank_difference = float(np.max(np.abs(original_rank - loaded_rank)))
assert probability_difference <= 1e-12
assert rank_difference <= 1e-12

artifacts = [
    model_path, classifier_weights_path, ranker_weights_path, predictions_path,
    feature_contract_path, metrics_path, quantity_plot, calibration_plot, risk_map, importance_plot,
]
manifest_path = OUTPUT_DIR / "run_manifest.json"
write_json({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": "champion",
    "source_stage": TRAINING_DATA_STAGE,
    "software": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__, "lightgbm": lgb.__version__,
    },
    "hardware": {"reported_gpu": GPU_NAME, "lightgbm_device": DEVICE},
    "inputs": {
        "all_parquet": {"path": ALL_PARQUET, "sha256": sha256(ALL_PARQUET)},
        "fire_region_csv": (
            {"path": FIRE_REGION_CSV_PATH, "sha256": sha256(FIRE_REGION_CSV_PATH)}
            if FIRE_REGION_CSV_PATH is not None else None
        ),
        "meta_json": {"path": META_JSON, "sha256": sha256(META_JSON)},
        "dataset_metadata_json": {"path": DATASET_METADATA_JSON, "sha256": sha256(DATASET_METADATA_JSON)},
        "feature_columns_json": {"path": FEATURE_COLUMNS_JSON, "sha256": sha256(FEATURE_COLUMNS_JSON)},
    },
    "artifacts": {
        path.name: {"path": path, "size_bytes": path.stat().st_size, "sha256": sha256(path)}
        for path in artifacts
    },
    "reload_check": {
        "rows": len(verification),
        "maximum_classifier_difference": probability_difference,
        "maximum_ranker_difference": rank_difference,
    },
}, manifest_path)

print("Training outputs for the inference notebook:")
for path in (model_path, classifier_weights_path, ranker_weights_path, feature_contract_path, metrics_path, manifest_path):
    print(f"  {path}")
print(f"Artifact reload passed on {len(verification):,} rows with zero difference.")


## 11. Kaggle handoff to inference

1. Run this notebook completely (`full` mode).
2. Confirm the artifact exists under the run output dir, e.g.  
   `/kaggle/working/champion_training_outputs_<stage>_<subset>/models/champion_model.joblib`.
3. Use **Save Version** in Kaggle and expose the notebook outputs as a dataset.
4. Open `Champion_Wildfire_Inference.ipynb` and attach:
   - that training-output dataset;
   - matching history:  
     `.../california-wildfire-knn` or `.../california-wildfire-median`.
5. Set `HISTORY_DATA_DIRECTORY` if needed (defaults to the KNN pack). `firms_test` is not required at inference.

The inference notebook never calls `.fit()`; all model weights come from this training artifact.
